# TURING-1 quickstart
Instalar antes: `pip install git+https://TU_REPO_URL.git` (o `pip install .` local). Solo requiere `torch`.

In [ ]:
import torch
import turing
print(turing.__version__, torch.__version__)

In [ ]:
model = turing.TuringForCausalLM.from_preset("tiny")
model.eval()
n = sum(p.numel() for p in model.parameters())
print(f"params: {n/1e6:.2f}M")
ids = torch.randint(0, 256, (1, 16))
out = model(input_ids=ids, labels=ids)
print("logits:", tuple(out["logits"].shape), "loss:", round(out["loss"].item(), 3))

In [ ]:
with torch.no_grad():
    g = model.generate(ids, max_new_tokens=12)
    s, st = model.generate_speculative(ids, max_new_tokens=12, return_stats=True)
print("greedy == especulativo:", bool(torch.equal(g, s)))
print("accept:", st)

In [ ]:
from turing import TrainConfig, train_step
import torch as _t
_t.manual_seed(0)
opt = _t.optim.AdamW(model.parameters(), lr=3e-3)
cfg = TrainConfig(lr=3e-3, bias_lr=0.01)
data = _t.randint(2, 30, (4, 24))
for _ in range(10):
    st = train_step(model, opt, None, data, cfg)
print("loss:", round(st["loss"], 4), "bias vivo:",
      any(bool((ly.ffn.expert_bias != 0).any()) for ly in model.model.layers if ly.is_moe))

In [ ]:
from turing import encode_text, decode_ids, memory_report
ids = encode_text("hola mundo")
print(decode_ids(ids[0, :4]), "->", tuple(ids.shape))
r = memory_report(turing.get_preset("1.2b"))
print(f"1.2b: {r['params_total']/1e9:.2f}B total, KV {r['kv_turing_MB']:.0f}MB (-{100*r['kv_reduction']:.0f}%)")